# Suite TSP — The TypeScript Patterns dialect

ECMA-262's binding and assignment patterns (14.3.3, "Destructuring Binding Patterns"; 13.15.5, "Destructuring
Assignment"), the sublanguage a function's, an arrow's, a signature's and a function type's parameters and an
assignment's target hold: the dialect `Expressions.TypeScript.Patterns` of the TypeScript family, whose `Term` was
TypeScript5's `Pattern` union, its branches unchanged, its kinds mbse-programs' patterns, its holder, what a pattern
binds through what holds it, its editions, its printing, and its evaluation, which refuses each kind by name (mbse-basis [CAP-277: TypeScript's types and destructuring patterns as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-277.md);
[CAP-228: the TypeScript5 expression dialect](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-228.md)).

In [ ]:
import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Unicode from "@mbse/expressions/Unicode/Expressions";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as TypeScriptPatterns from "@mbse/expressions/TypeScript/Patterns";
import * as TypeScriptTypes from "@mbse/expressions/TypeScript/Types";
import { Domains as D, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";
import { Simplifiers as Simplification } from "@mbse/expressions/Framework";
const { NotEvaluated } = Simplification;

const { NotImplementedError, ValueError } = Errors;
const [TS, TV, TSText] = [TypeScript5.Expressions, TypeScript5.Simplifiers, TypeScript5.Text];
const [TSP, TSPText, TST] = [TypeScriptPatterns.Expressions, TypeScriptPatterns.Text, TypeScriptTypes.Expressions];

const store: any = new Builders.Store(); // the framework, the standard dialects TypeScript5's domains use, and its family
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, Unicode.install, TypeScript5.install,
  TypeScriptTypes.install, TypeScriptPatterns.install]) {
  store.transaction(install);
}
store.transaction(TypeScriptPatterns.define); // the versions of TypeScript 5 the Patterns dialect covers
const READER = new Map([TS.DIALECT, TST.DIALECT, TSP.DIALECT].map((d) => [d.union, d])); // the dialect whose union holds a term

/** The holder of the term `spec` writes, `dialect`'s own (the Patterns dialect's unless given), written in a
 * transaction. */
function held(spec: unknown, dialect: F.Dialect = TSP.DIALECT): any {
  return store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec))));
}

const pattern = (spec: unknown): any => F.term(held(spec)); // a term of the Patterns dialect, held by its holder
const ts = (spec: unknown): any => F.term(held(spec, TS.DIALECT)); // a term of TypeScript5, held by its holder
const names = (found: Iterable<string>): string[] => [...found].sort();

/** A spec of the identifier `text`, in any union that has identifiers. */
function name(text: string): (e: any) => any {
  return (e: any) => e.identifier((i: any) => i.name(text));
}

const one = (e: any) => e.literal((li: any) => li.value((v: any) => v.as_scalar(1.0)));
// [a, , ...r]: a pattern of every kind but a parameter property, and the terms a target may be
const every = (p: any) => p.objectpattern((o: any) => o.properties([ // { k: [a, , ...r] = c, m: x.y, ...z }
  (q: any) => q.property((s: any) => s.propertyKind("init").key(name("k")).value((v: any) => v.assignmentpattern((a: any) => a.left(
    (t: any) => t.arraypattern((r: any) => r.elements([name("a"), (e: any) => e.elision((h: any) => h), (e: any) => e.restelement(
      (s: any) => s.argument(name("r")))]))).right(name("c"))))),
  (q: any) => q.property((s: any) => s.propertyKind("init").key(name("m")).value((v: any) => v.memberexpression(
    (m: any) => m.object(name("x")).property(name("y"))))),
  (q: any) => q.restelement((r: any) => r.argument(name("z")))]));

<a id="TSP-01"></a>
## TSP-01 · The Patterns dialect's union, its kinds' schemas in mbse-programs' shapes, the properties that hold it as a subterm, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML

In [ ]:
// Its Term: its four patterns, the terms of TypeScript5 a target may be, a parameter property and invalid syntax, as
// TypeScript5's `Pattern` union had them; each own kind's schema is named by its path; it has no helper of its own.
const union: string[] = store.registered(TSP.DIALECT.union).branches.map((b: any) => b.name);
assert(same(union, [...TSP.TAGS.slice(0, 4), ...TSP.TERMS, "tsparameterproperty", "error", "missing"]) && TSP.HELPERS.length === 0);
assert(same(TSP.TAGS, ["arraypattern", "objectpattern", "assignmentpattern", "restelement", "tsparameterproperty"]));
assert(TSP.DIALECT.name() === "TypeScript Patterns" && TSP.DIALECT.tag === "typescript_patterns");
assert(TSP.DIALECT.path === "TypeScript.Patterns" && TSP.DIALECT.union === "Expressions.TypeScript.Patterns.Term");
const Patterns: any = store.registered(TSP.DIALECT.union);
const kinds = new Map<string, any>(Patterns.branches.map((b: any) => [b.name, b.schema]));
assert(same(TSP.TAGS.map((tag) => kinds.get(tag).name), ["ArrayPattern", "ObjectPattern", "AssignmentPattern", "RestElement",
  "TSParameterProperty"].map((kind) => `Expressions.TypeScript.Patterns.${kind}`)));
assert(TSP.TERMS.every((tag) => kinds.get(tag).name.startsWith("Expressions.TypeScript.TypeScript5."))); // TypeScript5's
assert(equal(new Map(TSP.TAGS.map((tag) => [tag, [...kinds.get(tag).properties.keys()]])), map({
  arraypattern: ["decorators", "elements", "optional", "typeAnnotation"],
  objectpattern: ["decorators", "properties", "optional", "typeAnnotation"],
  assignmentpattern: ["decorators", "left", "optional", "typeAnnotation", "right"],
  restelement: ["decorators", "argument", "optional", "typeAnnotation", "value"],
  tsparameterproperty: ["decorators", "accessibility", "static", "override", "readonly", "parameter"] })));
assert(equal(F._parts(kinds.get("arraypattern")), [[], ["decorators", "elements"], ["optional", "typeAnnotation"]]));
assert(equal(F._parts(kinds.get("assignmentpattern")), [["left", "right"], ["decorators"], ["optional", "typeAnnotation"]]));
assert(kinds.get("assignmentpattern").properties.get("right").schema === store.registered(TS.DIALECT.union)); // a default, an expression
assert(kinds.get("restelement").properties.get("value").schema === kinds.get("assignmentpattern")); // a helper, as mbse-programs has it
// None refines Let: a pattern binds through what holds it
assert(!TSP.TAGS.some((tag) => F.isBinder(kinds.get(tag))));
// Where it is held: a function's, an arrow's, a signature's, a function type's and an index signature's parameters, an
// assignment's target; and TypeScript5's Term lists three of its kinds, a property's value holding them
const expressions = new Map<string, any>(store.registered(TS.DIALECT.union).branches.map((b: any) => [b.name, b.schema]));
const types = new Map<string, any>(store.registered(TST.DIALECT.union).branches.map((b: any) => [b.name, b.schema]));
const signatures = new Map<string, any>(store.registered("Expressions.TypeScript.Types.TypeElement").branches.map((b: any) => [b.name, b.schema]));
for (const [schema, prop] of [[expressions.get("functionexpression"), "params"], [expressions.get("arrowfunctionexpression"), "params"],
  [expressions.get("tsemptybodyfunctionexpression"), "params"], [types.get("tsfunctiontype"), "params"],
  [types.get("tsconstructortype"), "params"], [signatures.get("tsmethodsignature"), "params"],
  [signatures.get("tscallsignaturedeclaration"), "params"], [signatures.get("tsconstructsignaturedeclaration"), "params"],
  [signatures.get("tsindexsignature"), "parameters"]] as [any, string][]) {
  assert(schema.properties.get(prop).schema.element === Patterns, schema.name);
}
assert(expressions.get("assignmentexpression").properties.get("left").schema === Patterns);
assert(same(TSP.TAGS.slice(0, 3).map((tag) => expressions.get(tag)), TSP.TAGS.slice(0, 3).map((tag) => kinds.get(tag))));
// Its holder, refining the framework's Expression, its root narrowed to its Term: validated, linked by HasRoot,
// round-tripped
const Holder: any = store.registered("Expressions.TypeScript.Patterns.Expression");
assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
assert(Holder.properties.get("root").schema === Patterns && same([...Holder.adjacencies.keys()], ["abi"]) && Holder.validate().length === 0);
const target = held(every);
assert(Validators.Validate(store)(Holder, target).length === 0);
const linked = store.transaction(() => store.Schemas.OfExpression((s: any) => s.name("tsp.every").expression(
  (a: any) => a.insert((e: any) => e.expression(target)))));
assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((entry: any) => F.same(entry.expression, target)), [true]));
for (const [encode, decode] of [[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as any[]) {
  assert(F.same(decode(Holder, encode(Holder, target)).root, target.root));
  assert(F.same(decode(Patterns, encode(Patterns, F.term(target))), F.term(target)));
}
// Its install runs before or after its family's: each names the others' unions and helpers, created where not installed
const alone: any = new Builders.Store();
for (const install of [Framework.install, TypeScriptPatterns.install, TypeScript5.install, TypeScriptTypes.install]) {
  alone.transaction(install);
}
assert(same(alone.registered(TSP.DIALECT.union).branches.map((b: any) => b.name), union));

<a id="TSP-02"></a>
## TSP-02 · Each kind's checks; forms and `make`; what a pattern binds, through what holds it: a function's parameters within it, an assignment's target a reference; simplification leaves each kind in the residual, named

In [ ]:
// Each kind's checks: the subterms it needs, the identifiers within it, its annotation's type
for (const [spec, problems] of [
  [(p: any) => p.assignmentpattern((a: any) => a.left(name("a"))), ["an assignmentpattern needs a right"]],
  [(p: any) => p.restelement((r: any) => r), ["a restelement needs an argument"]],
  [(p: any) => p.tsparameterproperty((t: any) => t.accessibility("private")), ["a tsparameterproperty needs a parameter"]],
  [(p: any) => p.arraypattern((a: any) => a.elements([name("a b")])), ["elements 0: an identifier's name cannot be 'a b'"]],
  [(p: any) => p.objectpattern((o: any) => o.typeAnnotation((a: any) => a.typeAnnotation((t: any) => t.tsuniontype(
    (u: any) => u.types([(k: any) => k.tsanykeyword((h: any) => h)]))))), ["typeAnnotation typeAnnotation: a tsuniontype needs two types"]], // its
  // annotation's type, by its path
  [(p: any) => p.arraypattern((a: any) => a.elements([(e: any) => e.error((r: any) => r.text("1 +"))])), ["elements 0: invalid syntax '1 +'"]], // its
  // containers' terms, by container
] as [unknown, string[]][]) {
  assert(same(TSP.DIALECT.validate(pattern(spec), ["a"]), problems), globalThis.JSON.stringify([TSP.DIALECT.validate(pattern(spec), ["a"]), problems]));
}
// Its kinds read and written by their forms: a pattern's containers, its decorators and its elements, each its size
for (const node of F.walk(pattern(every))) {
  const reader = READER.get(F.dialectOf(node))!; // made by the dialect whose union holds it, in its holder
  assert(F.same(F.term(held(reader.make(F.form(node)), reader)), node));
}
const decorated = pattern((p: any) => p.arraypattern((a: any) => a.decorators([(d: any) => d.expression(name("d"))]).elements([name("a"), name("b")])));
const form = F.form(decorated);
assert(form.kind === "arraypattern" && same([...form.attributes].map(([key, part]) => [key, (part as F.Part).kind,
  (part as F.Part).attributes.size, (part as F.Part).size].join(" ")), ["decorators decorators 0 1", "elements elements 0 2"]));
assert(same(form.arguments.map((a) => F.nameOf(a)), ["d", "a", "b"]) && F.same(F.term(held(TSP.DIALECT.make(form))), decorated));
raises(ValueError, () => held(TSP.DIALECT.make(new F.Form("arraypattern", form.attributes, form.arguments.slice(0, 2)))),
  "an arraypattern's decorators, elements hold 3 arguments, got 2");
// What a pattern binds: the names of its identifiers, within arrays, objects, defaults, rests and parameter properties;
// a member, or any other target, binds none
assert(same(names(TSP.boundNames(pattern(every))), ["a", "r", "z"]) && TSP.boundNames === TS.boundNames);
assert(same(names(TSP.boundNames(pattern((p: any) => p.tsparameterproperty((t: any) => t.parameter((a: any) => a.assignmentpattern(
  (s: any) => s.left(name("q")).right(one))))))), ["q"]));
assert(TSP.boundNames(pattern((p: any) => p.identifier((i: any) => i))).size === 0); // a name not written
assert(TSP.boundNames(pattern((p: any) => p.memberexpression((m: any) => m.object(name("x")).property(name("y"))))).size === 0);
// A pattern binds through what holds it: a function's parameters within it, its defaults among it; an assignment's
// target is a reference to what it assigns
const arrow = ts((u: any) => u.arrowfunctionexpression((a: any) => a.params([every, (p: any) => p.assignmentpattern((s: any) => s.left(name("b")).right(name("a")))]).body(
  (b: any) => b.binaryexpression((e: any) => e.left(name("a")).operator("+").right(name("b"))))));
assert(same(names(SY.free(arrow, TS.DIALECT)), ["c", "x"]) && TS.DIALECT.validate(arrow, ["c", "x"]).length === 0); // a default, a member's object
const assign = ts((u: any) => u.assignmentexpression((a: any) => a.left(every).operator("=").right(name("o"))));
assert(same(names(SY.free(assign, TS.DIALECT)), ["a", "c", "o", "r", "x", "z"]));
assert(same(TS.DIALECT.validate(assign, ["a", "r", "c", "x", "o"]), ["left: properties 2: argument: identifier 'z' is not bound"]));
assert(same(names(SY.free(pattern(every), TSP.DIALECT)), ["a", "c", "r", "x", "z"])); // a pattern alone, held by its holder: references
// Evaluation refuses each kind by its dialect's name, as TypeScript5's evaluator does; inference knows nothing of it
for (const spec of [every, (p: any) => p.restelement((r: any) => r.argument(name("a")))]) {
  raises(NotEvaluated, () => TV.OfAny(pattern(spec)).evaluate(), `TypeScript Patterns ${F.kindOf(pattern(spec))[0]} is not simplified`);
}
raises(NotEvaluated, () => TV.OfAny(pattern(name("a"))).evaluate(), "the root is not a literal: a is not bound"); // a name, TypeScript5's kind, by its scope
assert(TSP.DIALECT.infer(pattern((p: any) => p.restelement((r: any) => r.argument(name("a")))), { a: D.Anything }).name() === "any");

<a id="TSP-03"></a>
## TSP-03 · Its editions: the versions of TypeScript 5, 5.0 to 5.9, by their published names, each with every one of its own kinds

In [ ]:
const tsp = store.Instances.Expressions.TypeScript.Patterns;
const VERSIONS = Array.from({ length: 10 }, (_, minor) => `TypeScript 5.${minor}`);
assert(same([...FS.standards(tsp).keys()], VERSIONS) && TSP.SINCE.size === 0);
assert(equal(FS.properties(FS.find(tsp, "TypeScript 5.9")!), map({ bodies: ["com.microsoft"], revision: ["version", "5.9"] })));
assert(same(FS.kinds(FS.find(tsp, "TypeScript 5.0")!).map((s: any) => s.name), TSP.TAGS.map((tag) => kinds.get(tag).name))); // its own

<a id="TSP-04"></a>
## TSP-04 · Printed as TypeScript writes patterns, from its own holder and within TypeScript5's, by the family's printer

In [ ]:
assert(TSPText.ToText === TSText.ToText && same([...TSPText.TEXTS.keys()].sort(), [...TSP.TAGS].sort()));
for (const [spec, text] of [
  [every, "({ k: [a,, ...r] = c, m: x.y, ...z })"], // as a statement holds it
  [(p: any) => p.arraypattern((a: any) => a.decorators([(d: any) => d.expression(name("d"))]).elements([name("a"),
    (e: any) => e.elision((h: any) => h)]).optional(true).typeAnnotation((t: any) => t.typeAnnotation((k: any) => k.tsarraytype(
      (r: any) => r.elementType((n: any) => n.tsnumberkeyword((h: any) => h)))))), "@d [a,,]?: number[]"],
  [(p: any) => p.tsparameterproperty((t: any) => t.accessibility("public").readonly(true).parameter((a: any) => a.assignmentpattern(
    (s: any) => s.left(name("q")).right(one)))), "public readonly q = 1"],
  [(p: any) => p.restelement((r: any) => r.argument(name("a")).typeAnnotation((t: any) => t.typeAnnotation((k: any) => k.tsanykeyword((h: any) => h)))),
    "...a: any"],
  [(p: any) => p.missing((m: any) => m.kind("pattern")), ""],
] as [unknown, string][]) {
  assert(TSText.ToText(pattern(spec)) === text, globalThis.JSON.stringify([TSText.ToText(pattern(spec)), text]));
}
raises(ValueError, () => TSText.ToText(pattern((p: any) => p.error((e: any) => e))), "TypeScript Patterns error has no text to print"); // by
// its dialect's name
assert(TSText.ToText(ts((u: any) => u.assignmentexpression((a: any) => a.left(every).operator("=").right(name("o"))))) ===
  "({ k: [a,, ...r] = c, m: x.y, ...z } = o)"); // an assignment's target, as a statement holds it
assert(TSText.ToText(ts((u: any) => u.functionexpression((f: any) => f.params([(p: any) => p.tsparameterproperty((t: any) => t.accessibility(
  "private").parameter(name("a"))), (p: any) => p.restelement((r: any) => r.argument(name("b")))])))) === "(function(private a, ...b) {})");